# SigAlg's `L2.inner` method

In [13]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `L2.inner` method in SigAlg computes the *inner product* of two random variables in an $L^2$-space. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/l2/#sigalg.l2.L2.inner).

## Mathematical definition

Let $X, Y \in L^2(\Omega, \mathcal{F}, P)$ be two square-integrable random variables on a probability space $(\Omega, \mathcal{F}, P)$. The *inner product* of $X$ and $Y$ is defined as

$$
\langle X, Y \rangle \stackrel{\text{def}}{=} \int_\Omega XY \, dP = E(XY).
$$

This inner product makes $L^2(\Omega, \mathcal{F}, P)$ into a *Hilbert space*. It satisfies the following properties:

1. *Bilinearity*: For all $X, Y, Z \in L^2(\Omega, \mathcal{F}, P)$ and $a, b \in \mathbb{R}$,
   $$
   \langle aX + bY, Z \rangle = a\langle X, Z \rangle + b\langle Y, Z \rangle,
   $$
   $$
   \langle X, aY + bZ \rangle = a\langle X, Y \rangle + b\langle X, Z \rangle.
   $$

2. *Symmetry*: For all $X, Y \in L^2(\Omega, \mathcal{F}, P)$,
   $$
   \langle X, Y \rangle = \langle Y, X \rangle.
   $$

3. *Positive-definiteness*: For all $X \in L^2$,
   $$
   \langle X, X \rangle \geq 0,
   $$
   with equality if and only if $X = 0$ almost surely.

## API examples


### Basic inner products

We begin by setting up a probability space and creating an $L^2$ space.

In [14]:
from sigalg.core import ProbabilityMeasure, RandomVariable, SampleSpace, SigmaAlgebra
from sigalg.l2 import L2

Omega = SampleSpace().from_sequence(size=4)

F = SigmaAlgebra(sample_space=Omega, name="F").from_dict(
    {
        0: 0,
        1: 1,
        2: 0,
        3: 1,
    }
)

P = ProbabilityMeasure(sample_space=Omega).from_dict(
    {
        0: 0.1,
        1: 0.15,
        2: 0.45,
        3: 0.3,
    }
)

H = L2(sample_space=Omega, sig_alg=F, prob_measure=P)

print(H)

H = L2(Omega, F, P)

* Sample space 'Omega':
[0, 1, 2, 3]

* Sigma algebra 'F':
        atom ID
sample         
0             0
1             1
2             0
3             1

* Probability measure 'P':
        probability
sample             
0              0.10
1              0.15
2              0.45
3              0.30


Create two random variables in the $L^2$-space.

In [15]:
X = (
    RandomVariable(domain=Omega, name="X")
    .from_dict(
        {
            0: 2,
            1: -1,
            2: 2,
            3: -1,
        }
    )
    .with_probability_measure(P)
)

Y = (
    RandomVariable(domain=Omega, name="Y")
    .from_dict(
        {
            0: 3,
            1: 5,
            2: 3,
            3: 5,
        }
    )
    .with_probability_measure(P)
)

print(X)
print(Y)

Random variable 'X':
        X
sample   
0       2
1      -1
2       2
3      -1
Random variable 'Y':
        Y
sample   
0       3
1       5
2       3
3       5


Compute the inner product $\langle X, Y \rangle$, and compare it to the expected value $E(XY)$ using the `expectation` method.

In [16]:
inner_XY = H.inner(X, Y)
expectation_XY = (X * Y).expectation().item()

print(f"<X, Y> = {inner_XY:.2f}")
print(f"E(XY) = {expectation_XY:.2f}")

<X, Y> = 1.05
E(XY) = 1.05


### Symmetry

The inner product is symmetric: $\langle X, Y \rangle = \langle Y, X \rangle$.

In [17]:
inner_YX = H.inner(Y, X)
print(f"<X, Y> = {inner_XY:.2f}")
print(f"<Y, X> = {inner_YX:.2f}")

<X, Y> = 1.05
<Y, X> = 1.05


### Bilinearity

The inner product is bilinear: $\langle aX + bY, Z \rangle = a\langle X, Z \rangle + b\langle Y, Z \rangle$.

In [18]:
Z = RandomVariable(domain=Omega, name="Z").from_dict(
    {
        0: 1,
        1: -2,
        2: 1,
        3: -2,
    }
)

a, b = 2.5, -1.3

# Left side: ⟨aX + bY, Z⟩
left = H.inner(a * X + b * Y, Z)

# Right side: a⟨X, Z⟩ + b⟨Y, Z⟩
right = a * H.inner(X, Z) + b * H.inner(Y, Z)

print(f"<{a}X + {b}Y, Z> = {left:.6f}")
print(f"{a}<X, Z> + {b}<Y, Z> = {right:.6f}")

<2.5X + -1.3Y, Z> = 8.705000
2.5<X, Z> + -1.3<Y, Z> = 8.705000


### Cauchy-Schwarz inequality

The Cauchy-Schwarz inequality states that $|\langle X, Y \rangle| \leq \|X\| \cdot \|Y\|$:

In [19]:
inner_XY = H.inner(X, Y)
norm_X = H.norm(X)
norm_Y = H.norm(Y)

print(f"|<X, Y>| = {abs(inner_XY):.6f}")
print(f"||X|| · ||Y|| = {norm_X * norm_Y:.6f}")

|<X, Y>| = 1.050000
||X|| · ||Y|| = 6.552099


### Connection to covariance

The *covariance* of two random variables can be expressed using the inner product:

$$
\text{Cov}(X, Y) = \langle X - E(X), Y - E(Y) \rangle.
$$

Let's compute the covariance using both the `cov` method and the inner product.

In [20]:
from sigalg.core import Operators

E = Operators.expectation
cov = Operators.cov

# Using the cov method
X.prob_measure = P
Y.prob_measure = P
cov_XY = cov(X, Y).item()

# Using the inner product
X_centered = X - E(X)
Y_centered = Y - E(Y)
cov_XY_inner = H.inner(X_centered, Y_centered)

print(f"Cov(X, Y) using cov method: {cov_XY:.6f}")
print(f"Cov(X, Y) using inner product: {cov_XY_inner:.6f}")

Cov(X, Y) using cov method: -1.485000
Cov(X, Y) using inner product: -1.485000
